# 13. 판정요지의 제품별 판단 근거 추출 및 사례 검색 비교

12번 실험에서 판정요지 문장과 지식베이스 문서의 SBERT 유사도가 전반적으로 높게 나타났고, 단순 임계값 제거 방식은 잔여 문장을 거의 남기지 못했습니다. 이번 실험은 판정요지에서 제품 속성과 관련되면서 지식베이스 문서와 상대적으로 덜 중복되는 문장을 후보로 추출합니다.

**주의:** 자동 추출 결과는 제품별 판단 근거의 확정 정답이 아닙니다. 문장별 유사도와 규칙 기반 특징으로 만든 후보이며, 표본 검토가 필요합니다. 검색 평가는 동일한 4-fold StratifiedGroupKFold를 사용합니다.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('sentence_transformers') is None: subprocess.check_call([sys.executable,'-m','pip','-q','install','sentence-transformers'])
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('사용 장치:',DEVICE)


## 1. 데이터 불러오기


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
target_col,summary_col = 'HS Code','Preprocessed Ruling Summary'
required = [target_col]+input_cols+[summary_col]
missing = [c for c in required if c not in data_all.columns]
if missing: raise ValueError(f'필수 컬럼 누락: {missing}')
data_all[required] = data_all[required].astype(str).replace('nan','').fillna('')
data_all[target_col] = data_all[target_col].str.strip()
data_all['query_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(target_col).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(target_col)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[target_col].isin(labels)].copy().reset_index(drop=True)
print('사례:',len(data),'| HS 코드:',data[target_col].nunique())
display(data[[target_col]+input_cols+[summary_col]].head(3))


## 2. 문장 분리 및 SBERT 유사도 계산

각 판정요지 문장에 대해 (1) 지식베이스 문서와의 최대 유사도, (2) 해당 제품의 5개 속성을 합친 질의와의 유사도를 계산합니다. 제품 속성과 관련성이 높고 지식베이스와의 중복도가 상대적으로 낮은 문장을 제품별 판단 근거 후보로 봅니다.


In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME,device=DEVICE)
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
kb_texts = kb['document_text'].astype(str).tolist()
kb_emb = model.encode(['passage: '+x for x in kb_texts],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True,device=DEVICE)
def split_sentences(text):
    text = re.sub(r'\\s+',' ',str(text)).strip()
    parts = re.split(r'(?<=[.!?。])\\s+|(?<=다\\.)\\s*',text)
    parts = [x.strip(' ,;:') for x in parts if len(x.strip())>=12]
    if len(parts)<=1 and len(text)>350: parts = [text[i:i+250] for i in range(0,len(text),220)]
    return parts if parts else ([text] if text else [])
sentence_rows = []
for row_id,row in data.iterrows():
    for sent_id,sent in enumerate(split_sentences(row[summary_col])):
        sentence_rows.append({'row_index':row_id,'sentence_id':sent_id,'sentence':sent,'query_text':row['query_text']})
sentences = pd.DataFrame(sentence_rows)
sent_emb = model.encode(['query: '+x for x in sentences['sentence'].tolist()],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True,device=DEVICE)
query_emb_for_sent = model.encode(['query: '+x for x in data['query_text'].astype(str).tolist()],batch_size=64,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True,device=DEVICE)
kb_sim = sent_emb@kb_emb.T
sentences['kb_similarity'] = kb_sim.max(axis=1)
sentences['matched_knowledge_id'] = kb.iloc[kb_sim.argmax(axis=1)]['knowledge_id'].to_numpy()
sentences['matched_source_sheet'] = kb.iloc[kb_sim.argmax(axis=1)]['source_sheet'].to_numpy()
sentence_query_idx = sentences['row_index'].to_numpy()
sentences['attribute_similarity'] = np.sum(sent_emb*query_emb_for_sent[sentence_query_idx],axis=1)
sentences['specificity_score'] = sentences['attribute_similarity']-sentences['kb_similarity']
display(sentences.sort_values('specificity_score',ascending=False).head(20))


## 3. 제품별 판단 근거 후보 추출

후보 조건은 제품 속성 유사도 0.35 이상이며, 지식베이스 최대 유사도가 각 임계값보다 낮은 문장입니다. 0.72, 0.78, 0.84를 비교해 임계값에 따른 추출량과 검색 성능 변화를 확인합니다. 이 기준은 탐색적 규칙이며 법적 판단의 정답 라벨이 아닙니다.


In [ ]:
KB_THRESHOLDS = [0.72,0.78,0.84]
ATTRIBUTE_MIN = 0.35
for threshold in KB_THRESHOLDS:
    keep = (sentences['attribute_similarity']>=ATTRIBUTE_MIN)&(sentences['kb_similarity']<threshold)
    selected = sentences.loc[keep].groupby('row_index')['sentence'].apply(' '.join)
    data[f'extracted_ruling_{threshold}'] = data.index.map(selected).fillna('')
    data[f'extracted_case_{threshold}'] = (data['query_text']+' '+data[f'extracted_ruling_{threshold}']).str.strip()
extraction_rows = []
for threshold in KB_THRESHOLDS:
    keep = (sentences['attribute_similarity']>=ATTRIBUTE_MIN)&(sentences['kb_similarity']<threshold)
    extraction_rows.append({'kb_threshold':threshold,'attribute_min':ATTRIBUTE_MIN,'selected_sentence_n':int(keep.sum()),'selected_sentence_ratio':float(keep.mean()),'cases_with_extracted_text':int((data[f'extracted_ruling_{threshold}'].str.len()>0).sum()),'case_n':len(data)})
extraction_metrics = pd.DataFrame(extraction_rows)
display(extraction_metrics)
display(data[[target_col,summary_col,'extracted_ruling_0.78']].head(12))


## 4. 사례 검색 성능 비교

검색 문서에는 각 Fold의 학습 사례만 포함하고, 테스트 제품의 질의에는 5개 제품 속성만 사용합니다. 전체 사례 문서 임베딩과 테스트 질의 임베딩은 각각 한 번만 계산해 Fold별 인덱싱으로 재사용합니다. 따라서 Fold마다 같은 문장을 반복 임베딩하지 않습니다.


In [ ]:
def retrieval_metrics(sim,relevant):
    order = np.argsort(sim)[::-1]
    rel = np.asarray(relevant,dtype=int)[order]
    hits = np.where(rel==1)[0]
    out = {'mrr':1/(hits[0]+1) if len(hits) else 0.0}
    for k in (5,10): out[f'recall@{k}'] = float(rel[:k].sum()/max(rel.sum(),1))
    gains = rel[:10]/np.log2(np.arange(2,12))
    ideal = np.sort(rel)[::-1][:10]/np.log2(np.arange(2,12))
    out['ndcg@10'] = float(gains.sum()/ideal.sum()) if ideal.sum()>0 else 0.0
    return out
variant_cols = {'product_only':'query_text','full_ruling':'full_case','extracted_0.72':'extracted_case_0.72','extracted_0.78':'extracted_case_0.78','extracted_0.84':'extracted_case_0.84'}
data['full_case'] = (data['query_text']+' '+data[summary_col]).str.strip()
doc_embeddings = {}
for variant,doc_col in variant_cols.items():
    doc_embeddings[variant] = model.encode(['passage: '+x for x in data[doc_col].astype(str).tolist()],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True,device=DEVICE)
all_query_embeddings = model.encode(['query: '+x for x in data['query_text'].astype(str).tolist()],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True,device=DEVICE)
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
eval_rows,fold_rows = [],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
    train_labels = data.iloc[tr][target_col].astype(str).to_numpy()
    overlap = len(set(data.iloc[tr]['duplicate_group'])&set(data.iloc[te]['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':overlap})
    for variant in variant_cols:
        train_emb = doc_embeddings[variant][tr]
        for idx in te:
            relevant = train_labels==str(data.iloc[idx][target_col])
            metrics = retrieval_metrics(train_emb@all_query_embeddings[idx],relevant)
            eval_rows.append({'fold':fold,'variant':variant,**metrics})
    print(f'Fold {fold}/{N_SPLITS} 완료')
fold_audit = pd.DataFrame(fold_rows)
retrieval_results = pd.DataFrame(eval_rows)
retrieval_summary = retrieval_results.groupby('variant')[['mrr','recall@5','recall@10','ndcg@10']].mean().reset_index()
display(fold_audit)
display(retrieval_summary.sort_values('mrr',ascending=False))


## 5. 추출 기준 민감도 및 결과 저장

추출 임계값에 따라 선택되는 문장과 검색 성능이 어떻게 달라지는지 확인합니다. 동일한 Fold에서 여러 기준을 비교하므로 최적 기준을 확정하려면 별도 검증이 필요합니다.


In [ ]:
threshold_summary = retrieval_summary[retrieval_summary['variant'].str.startswith('extracted_')].copy()
threshold_summary['kb_threshold'] = threshold_summary['variant'].str.replace('extracted_','',regex=False)
display(threshold_summary)
sentences.to_csv('13_sentence_features.csv',index=False,encoding='utf-8-sig')
extraction_metrics.to_csv('13_extraction_metrics.csv',index=False,encoding='utf-8-sig')
retrieval_results.to_csv('13_retrieval_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('13_fold_audit.csv',index=False,encoding='utf-8-sig')
retrieval_summary.to_csv('13_retrieval_summary.csv',index=False,encoding='utf-8-sig')
print('13번 실험 결과 저장 완료')


## 6. 해석 유의사항

- `13_sentence_features.csv`: 문장별 지식베이스 유사도, 제품 속성 유사도 및 매칭 문서
- `13_extraction_metrics.csv`: 기준별 추출 문장 수와 사례별 추출 여부
- `13_retrieval_metrics.csv`: Fold·사례별 검색 지표
- `13_fold_audit.csv`: Fold 간 중복 그룹 누수 점검
- `13_retrieval_summary.csv`: 검색 방식별 평균 성능

제품 속성 유사도와 지식베이스 유사도의 차이는 제품별 고유 판단 근거의 정답 라벨이 아닙니다. 추출된 문장을 사람이 검토해 실제 제품 적용 판단인지 확인해야 합니다. 검색 지표는 사례 검색 품질이며 HS 코드 분류 성능과 동일하지 않습니다.
